In [3]:
# ===============================
# Fraud Detection Project
# ===============================

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from imblearn.over_sampling import SMOTE

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
df = pd.read_csv("fraud_detection.csv")

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
df = df[['type', 'amount', 'oldbalanceOrg',
         'newbalanceOrig', 'oldbalanceDest',
         'newbalanceDest', 'isFraud']]

df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()

In [ ]:
encoder = LabelEncoder()
df['type'] = encoder.fit_transform(df['type'])

# Save encoder
pickle.dump(encoder, open("encoder.pkl", "wb"))

df.head()

In [ ]:
X = df.drop("isFraud", axis=1)
y = df["isFraud"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("After SMOTE:", X_train.shape)

In [ ]:
# Initialize Models
ada = AdaBoostClassifier(random_state=42)
lr = LogisticRegression(max_iter=1000)
nb = MultinomialNB()

# Train Models
ada.fit(X_train, y_train)
lr.fit(X_train, y_train)
nb.fit(X_train, y_train)

print("Models Trained Successfully")

In [ ]:
def evaluate_model(model, name):
    predictions = model.predict(X_test)
    
    print(f"\n{name} Performance")
    print("Accuracy :", accuracy_score(y_test, predictions))
    print("Precision:", precision_score(y_test, predictions))
    print("Recall   :", recall_score(y_test, predictions))
    print("F1 Score :", f1_score(y_test, predictions))


evaluate_model(ada, "AdaBoost")
evaluate_model(lr, "Logistic Regression")
evaluate_model(nb, "Naive Bayes")

In [ ]:
# ==============================
# Confusion Matrix + Metrics
# ==============================

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_and_plot(model, model_name):
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"\n==============================")
    print(f"{model_name} Evaluation")
    print(f"==============================")
    
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))
    
    # Plot Confusion Matrix
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, 
                annot=True, 
                fmt="d", 
                cmap="Blues",
                xticklabels=["Genuine", "Fraud"],
                yticklabels=["Genuine", "Fraud"])
    
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()


# 🔹 Call for Each Model
evaluate_and_plot(ada, "AdaBoost")
evaluate_and_plot(lr, "Logistic Regression")
evaluate_and_plot(nb, "Naive Bayes")

In [ ]:
# ==============================
# ROC Curve + AUC Score
# ==============================

from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(7,6))

def plot_roc(model, model_name):
    
    # Get probability scores
    y_prob = model.predict_proba(X_test)[:,1]
    
    # Calculate ROC values
    fpr, tpr, threshold = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    # Plot
    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {roc_auc:.3f})")

# Plot for each model
plot_roc(ada, "AdaBoost")
plot_roc(lr, "Logistic Regression")
plot_roc(nb, "Naive Bayes")

# Diagonal reference line
plt.plot([0,1], [0,1], 'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

In [ ]:
import pickle

# Save Encoder
with open("encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Encoder Saved Successfully")

In [ ]:
# Save Model
with open("AdaBoostClassifier.pkl", "wb") as f:
    pickle.dump(ada, f)

print("Model Saved Successfully")